# Mining comparison, Phase 4b: grouping × token granularity (AIT-ADS)

On fox under cscas_grouping, the atemporal methods' features mostly restate a single signature. Is that a property of the grouping (one group = one signature), or of the item granularity (`short`/`sig` stand in for the signature)? And does mining find signal beyond one signature once either changes?

- **Groupings:** cscas_grouping (reference), fixed_window, time_delta, AlertBERT. DeepCASE is left out: its groups are single-code clusters, and 81% of alerts sit in mixed groups.
- **Token arms:**
  - **T0:** full codes
  - **T1:** + `short_src`, `short_cat`
  - **T2:** + `short_family`, `sig_family` (abstraction map)
  - **T3:** coarse levels only, without `short`/`sig`
- **Setup per cell:** every grouping is cut at the same moment (the cscas_grouping 70% point); precision gate ≥ 2π for attack and ≤ π for benign; K = 50 per class; features with identical firing count once; Eclat's count floor is relative to the training attacks. Temporal methods run on T0 only.
- **Restatement**, two reported metrics (PASS = live features for both classes):
  - **confined:** every mining-window group the feature fires on contains one short code *s*, so it can never fire outside *s*. Exact for single-code groupings, but under multi-code groupings a frequent background code co-occurring in all those groups also counts.
  - **equivalent:** the feature fires on (almost) exactly the groups containing *s*, i.e. Jaccard ≥ 0.9. It means the same under every grouping.
- **Unseen-code coverage:** the share of test attack groups with **at least one** code never seen in the mining window that are hit by at least one attack feature (`*_all`: all codes unseen).

Logic: `thesis.system_eval.mining_mini_experiment`. Run: `scripts/system_eval/run_grouping_token_grid.py`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from thesis.paths import ROOT

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

RUN_DIRS = None  # None -> latest run per scenario, else a list of paths
BASE = ROOT / "artifacts" / "experiments" / "mining_grouping_tokens"

def _latest_per_scenario() -> list[Path]:
    latest: dict[str, Path] = {}
    for d in sorted(x for x in BASE.glob("*") if (x / "summary.csv").exists()):
        s = pd.read_csv(d / "summary.csv", usecols=["window"])
        if "equivalent_signature_share" not in pd.read_csv(d / "summary.csv", nrows=1).columns:
            continue  # runs before the equivalence metric
        latest[s["window"].iloc[0].split()[0]] = d
    return list(latest.values())

run_dirs = [Path(p) for p in RUN_DIRS] if RUN_DIRS else _latest_per_scenario()
summary = pd.concat([pd.read_csv(d / "summary.csv").assign(run=d.name) for d in run_dirs], ignore_index=True)
features = pd.concat([pd.read_csv(d / "features.csv").assign(run=d.name) for d in run_dirs], ignore_index=True)
summary["scenario"] = summary["window"].str.split().str[0]
features = features.merge(summary[["run", "grouping", "token_arm", "method", "scenario"]].drop_duplicates(),
                          on=["run", "grouping", "token_arm", "method"], how="left")
GROUPINGS = [g for g in ["cscas_grouping", "fixed_window", "time_delta", "alertbert"] if g in set(summary.grouping)]
print({d.name: s for d, s in zip(run_dirs, summary.groupby("run").scenario.first())})

## The mining windows per scenario and grouping

In [ ]:
summary.groupby(["scenario", "grouping"])[["n_train", "n_train_attack", "n_test", "n_test_attack", "test_base_rate", "n_test_attack_unseen", "n_test_attack_unseen_all"]].first()

## Headline metrics: scenario × grouping × token arm, per method

In [ ]:
def pivot(metric: str) -> pd.DataFrame:
    p = summary.pivot_table(index=["method", "token_arm"], columns=["scenario", "grouping"], values=metric)
    return p.round(2)

for metric in ["single_signature_share", "equivalent_signature_share", "unseen_attack_coverage",
               "test_attack_coverage", "test_benign_coverage", "live_share_test"]:
    print(f"\n=== {metric}")
    display(pivot(metric))

## Verdicts and feature counts

In [ ]:
cols = ["scenario", "grouping", "token_arm", "method", "status", "pass_strict", "eclat_min_support_count",
        "n_raw", "n_gated_conditions", "n_gated", "n_budget_attack", "n_budget_benign", "mine_seconds"]
summary[[c for c in cols if c in summary]].sort_values(["scenario", "method", "grouping", "token_arm"])

## cscas_grouping: short code vs. alert name as the signature identity

In [ ]:
if "single_signature_share_name" in summary:
    display(summary.dropna(subset=["single_signature_share_name"])[
        ["scenario", "method", "token_arm", "single_signature_share", "single_signature_share_name"]])

## Attack features that are not equivalent to one signature, per cell (top 5 by support)

In [ ]:
nr = features[(features.source_label == "attack") & ~features.equivalent_signature]
for (sc, g, arm, m), grp in nr.groupby(["scenario", "grouping", "token_arm", "method"]):
    print(f"\n=== {sc} | {g} | {arm} | {m} ({len(grp)} features)")
    display(grp.sort_values("support", ascending=False).head(5)[
        ["condition", "train_fires", "train_attack_rate", "test_fires", "test_attack_rate", "single_signature", "equiv_jaccard"]])